# Objetivo
O objetivo desse notebook é o de ler os arquivos profile, transactions, offers, fazendo sua limpeza, usando o spark para essa finalidade

# 1. Importing Libraries

In [0]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import from_json
from pyspark.sql import DataFrame
from pyspark.sql.functions import struct, col

# 2. Defining variables 

In [0]:
# raw : raw data origin ; processed : data ready for analysis
raw_data =       "dbfs:/Workspace/Users/marcosferreira220863@gmail.com/ifood_chalenge/data/raw"
processed_data = "dbfs:/Workspace/Users/marcosferreira220863@gmail.com/ifood_chalenge/data/processed"

# 3. Using spark to load the data:
* transaction
* offers
* profile 

## 3.1. Transactions

In [0]:
####
# DBTITLE 1,Read transactions
transaction_file_name = "transactions.json" 
file_path = os.path.join(raw_data,transaction_file_name)
db_sp_transactions = spark.read.json(file_path)
display(db_sp_transactions)


## 3.2. Offers

In [0]:
####
# DBTITLE 2,Read offers
offers_file_name  = "offers.json"
file_path = os.path.join(raw_data,offers_file_name)
db_sp_offers = spark.read.json(file_path)
display(db_sp_offers)


## 3.3.Profile

In [0]:
# DBTITLE 3,Read profile
profile_file_name = "profile.json"
file_path = os.path.join(raw_data,profile_file_name)
db_sp_profile = spark.read.json(file_path)
display(db_sp_profile)

# 4. Exploring Data
## 4.1. Row Number

In [0]:
# transactions
# offer 
# profile 
row_count_transaction   = db_sp_transactions.count()
rows_count_offer        = db_sp_offers.count()
rows_count_profile      = db_sp_profile.count()
print(f"")
print("=========================") 
print(f"Transactions row count: {row_count_transaction}")
print(f"Offers row count: {rows_count_offer}")
print(f"Profile row count: {rows_count_profile}")
print(f"=========================")
print(f"")

## 4.2. Verifying Duplicates and remove, if exisits

In [0]:
def verify_duplicated(df):
    """
    verify_dpulicated: 
        verifica se há linhas duplicadas
    parametros:
        df: dataframe
    retorno:
        df_duplicadas: dataframe com as linhas duplicadas
    """
    df_duplicadas = df.groupBy(df.columns) \
    .count() \
    .filter(F.col("count") > 1)

# Exibe as linhas que possuem duplicatas e quantas vezes aparecem
    return df_duplicadas


#### a. transactions


In [0]:
dupl_transactions = verify_duplicated(db_sp_transactions)
display(dupl_transactions)


##### i. removing duplicates 

In [0]:
print(f"")
print(f"="*50)
print(f"linhas tb transactions antes da limpeza : {db_sp_transactions.count()}")
db_sp_transactions_clean = db_sp_transactions.dropDuplicates()
print(f"linhas tb transactions após a limpeza : {db_sp_transactions_clean.count()}")
print(f"="*50)
print(f"")



#### b. offer 

In [0]:
dupl_offers = verify_duplicated(db_sp_offers)
display(dupl_offers)

#### c. profile 

In [0]:
dupl_profile = verify_duplicated(db_sp_profile)
display(dupl_profile)

## 4.3. Eliminating profiles with age == 118 or gender null

### 4.3.1. Verifying the profiling of the table

In [0]:
def profile_data(df):
    """
    Função para gerar um resumo completo do dataframe:
    - Contagem de nulos por coluna
    - Percentual de nulos
    - Tipos de dados
    - Quantidade de valores únicos
    : param df: Dataframe a ser analisado
    : return: Dataframe com o resumo
    """
    total_rows = df.count()
    
    # Lista de tuplas com: Nome da Coluna, Tipo, Nulos, % Nulos, Valores Distintos
    profile_list = []
    
    for col_name in df.columns:
        null_count = df.filter(F.col(col_name).isNull()).count()
        null_pct = (null_count / total_rows) * 100
        distinct_count = df.select(col_name).distinct().count()
        dtype = dict(df.dtypes)[col_name]
        
        profile_list.append((col_name, dtype, null_count, null_pct, distinct_count))
    
    # Criar um DataFrame com o resumo
    columns = ["Coluna", "Tipo", "Total_Nulos", "Percentual_Nulos", "Valores_Unicos"]
    profile_df = spark.createDataFrame(profile_list, columns)
    
    return profile_df

In [0]:
# Uso da função
df_summary = profile_data(db_sp_profile)
display(df_summary)

Exisem 2175 registros com gênero igual a Null. Vamos analisar as colunas da tabela profiles.

#### a. distribuição de variáveis numéricas

In [0]:
def plot_histograms(df, columns):
    # Convertemos apenas as colunas necessárias para o driver
    # O sample(0.1) pega 10% dos dados para não sobrecarregar
    #pdf = df.select(columns).sample(0.1).toPandas()
    pdf = df.select(columns).toPandas()
    
    for col in columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(pdf[col], kde=True)
        plt.title(f"Distribuição da coluna: {col}")
        plt.show()



In [0]:
# Lista de colunas numéricas para plotar
num_cols = ["age", "credit_card_limit"]
plot_histograms(db_sp_profile, num_cols)

Analisando o histograma para a variável 'age' é possível identificar um outlier, que corresponde a idade == 118. A coluna 'credit_card_limit é desbalanceada( skill à direita), mas eu estou desconsideerando essa assimetria para efeitos de construção de modelo, apenas por limitação de tempo.

#### b. distribuição de variáveis categóricas

In [0]:
def plot_cat_histograms(df):
    """
    Função para plotar histogramas de colunas categóricas.
    : param df: Dataframe a ser analisado
    """
    # Obter colunas categóricas
    cat_cols = [col for (col, dtype) in df.dtypes if( (dtype == "string") and (col) != "id" ) ]
    
    # Plotar histogramas
    plot_histograms(df, cat_cols)


In [0]:
plot_cat_histograms(db_sp_profile)

A variável gender está razoavelmente balanceada, enquanto a variável age tem uma distribuição desbalanceada.

In [0]:
df_contagem = db_sp_profile.groupBy("gender") \
    .count() \
    .orderBy(F.col("count").desc())

df_contagem.show()

O gênero 'outros-O' é um gênero válido, no meu entender,e não vai ser removido

#### c. Ocorrência de valores faltantes em gênero com padrão MNAR(Missing not at random) 

In [0]:
df_flag = db_sp_profile.withColumn("age_status",
    F.when(F.col("age") == 118, "Idade 118")\
     .when(F.col("age").isNull(), "Idade Nula")\
     .otherwise("Idade Válida")
).withColumn("gender_status",\
    F.when(F.col("gender").isNull(), "Gênero Nulo")\
     .otherwise("Gênero Preenchido")
)

# 2. Agrupar para contar as combinações
# Fazemos o .collect() aqui porque o resultado é pequeno (apenas 4 ou 5 combinações)
df_plot = df_flag.groupBy("age_status", "gender_status") \
                 .count() \
                 .toPandas()

print(df_plot)

In [0]:
plt.figure(figsize=(10, 6))

# Usamos o dataframe Pandas criado no passo anterior
chart = sns.barplot(
    data=df_plot,
    x="age_status",
    y="count",
    hue="gender_status",
    palette="viridis" # Escolha uma paleta de cores profissional
)

plt.title('Relação entre Idade Suspeita (118) e Gênero Nulo', fontsize=15)
plt.xlabel('Status da Idade', fontsize=12)
plt.ylabel('Quantidade de Clientes', fontsize=12)
plt.legend(title='Status do Gênero')

# Adicionar os valores nas barras (opcional, mas ajuda na apresentação)
for container in chart.containers:
    chart.bar_label(container, fmt='%.0f')

plt.show()

A conclusão do gráfico e da tabela acima é que registros de profile cujo indivíduo está com gênero Nulo está associado com a idade == 118. Vamos filtrar esses registros

#### d. filtrando linhas da tabela profile com idade == 118 ou gênero Nullo

In [0]:
db_sp_profile_clean = db_sp_profile.filter(F.col("age") != 118)
db_sp_profile_clean = db_sp_profile_clean.dropna(subset=["gender"])      
db_sp_profile_clean = db_sp_profile_clean.dropDuplicates()
print(f"")
print(f"="*50)
print(f"Número de linhas da tb. profile antes: {db_sp_profile.count()}")
print(f"Número de linhas da tb. profile depois: {db_sp_profile_clean.count()}")
print(f"="*50)
print(f"")


# 5. Salvando os dados limpos no diretório data\processed

## 5.1. Salvando profile

In [0]:
db_sp_profile_clean.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.profile_clean")



## 5.2. Salvando transactions :
É necessário fazer uma limpeza no campo value,antes, normalizando a estrutura

In [0]:
db_sp_transactions_clean.printSchema()

In [0]:
df_sp_transactions_clean_2 = db_sp_transactions_clean.withColumn(
    "value",
    struct(
        col("value.`offer id`").alias("offer_id"),
        col("value.offer_id").alias("offer_id_original"),
        col("value.amount").alias("amount"),
        col("value.reward").alias("reward")
    )
)

In [0]:
display(df_sp_transactions_clean_2)

In [0]:
#db_sp_transactions_clean2 = (
#    db_sp_transactions_clean
#    .withColumn(
#        "value",
#        struct(
#            col("value.amount").alias("amount"),
#            col("value.`offer id`").alias("offer_id"),
#            col("value.reward").alias("reward")
#        )
#    )
#)

In [0]:
df_sp_transactions_clean_2.printSchema()

In [0]:
display(df_sp_transactions_clean_2)

In [0]:
df_sp_transactions_clean_2.write\
    .mode("overwrite")\
        .saveAsTable("workspace.default.transaction_clean")

## 5.3. Salvando Offer 
Mesmo problema 

In [0]:
db_sp_offers.printSchema()


In [0]:
#spark.sql("DROP TABLE IF EXISTS workspace.default.offers_clean")

In [0]:
db_sp_offers.write\
    .mode("overwrite")\
        .saveAsTable("workspace.default.offers_clean")